# ELECTRA + ScalarMix + DANN — LLM-judge clean dataset (from Drive)

Same data setup as `Electra_LLMJudge_CleanDataset_FromDrive.ipynb`, with **domain adversarial training**:

- **Train / val:** `train.csv`, `val.csv` — label `education_level_judge`, domain `source_dataset`
- **Test:** `test.csv` (in-domain)
- **OOD:** `ood_onestop.csv`, `ood_race-middle.csv`, `ood_race-high.csv` (judge labels, from Drive)

Two-phase training: frozen encoder → partial unfreeze + GRL. Saves model, metrics, confusion matrices to `DRIVE_OUT_DIR`.

**Note:** `domain_id` is only required for **train/val** (DANN). OOD CSVs (`onestop`, `race-*`) are held out of training and are **not** dropped during load — eval uses judge labels + text only.

In [ ]:
!pip install -q transformers scikit-learn torch pandas matplotlib seaborn tqdm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ── Paths & hyperparameters ──────────────────────────────────────────────
DRIVE_CLEAN_DIR = "/content/drive/MyDrive/beyond_flesch/clean_dataset"
DRIVE_OUT_DIR = "/content/drive/MyDrive/beyond_flesch/trail/electra_llm_judge_dann"

MODEL_NAME = "google/electra-large-discriminator"
TEXT_COL = "full_text"
LABEL_COL = "education_level_judge"
SOURCE_COL = "source_dataset"

MAX_LEN = 512
BATCH_SIZE = 4
RNG_SEED = 42
LABEL_SMOOTHING = 0.1
BALANCE_TRAIN = False
EVAL_LABELS = [0, 1, 2]

# DANN (gentle recipe)
GRL_LAMBDA_MAX = 0.15
DOMAIN_LOSS_ALPHA = 0.1
PHASE1_EPOCHS = 2
PHASE1_LR = 1e-3
PHASE2_EPOCHS = 1  # phase 2 often unstable; best checkpoint usually from phase 1
PHASE2_LR = 2e-5
PARTIAL_FREEZE_LAYERS = 8
WARMUP_STEPS = 100

label2id = {"elementary": 0, "middle": 1, "high": 2}
id2label = {v: k for k, v in label2id.items()}

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
from pathlib import Path
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from torch import Tensor
from torch.nn import Parameter, ParameterList
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

CLEAN_DIR = Path(DRIVE_CLEAN_DIR)
if not CLEAN_DIR.joinpath("train.csv").exists():
    raise FileNotFoundError(
        f"Missing train.csv under {CLEAN_DIR}. Upload clean_dataset CSVs to Drive."
    )

os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
CM_DIR = os.path.join(DRIVE_OUT_DIR, "confusion_matrices")
os.makedirs(CM_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RNG_SEED)
print("Device:", device)
print("CLEAN_DIR:", CLEAN_DIR)
print("OUT:", DRIVE_OUT_DIR)

In [ ]:
# ── ScalarMix + DANN ─────────────────────────────────────────────────────
class ScalarMix(nn.Module):
    def __init__(self, mixture_size: int, trainable: bool = True) -> None:
        super().__init__()
        self.scalar_parameters = ParameterList(
            [Parameter(torch.zeros(1), requires_grad=trainable) for _ in range(mixture_size)]
        )
        self.gamma = Parameter(torch.ones(1), requires_grad=trainable)

    def forward(self, tensors: List[torch.Tensor]) -> torch.Tensor:
        w = torch.nn.functional.softmax(torch.cat([p for p in self.scalar_parameters]), dim=0)
        w = torch.split(w, 1)
        return self.gamma * sum(weight * t for weight, t in zip(w, tensors))


def grl_lambda_schedule(progress: float) -> float:
    progress = float(min(1.0, max(0.0, progress)))
    return GRL_LAMBDA_MAX * (2.0 / (1.0 + math.exp(-10.0 * progress)) - 1.0)


class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx: Any, x: Tensor, lambda_: float) -> Tensor:
        ctx.lambda_ = float(lambda_)
        return x.view_as(x)

    @staticmethod
    def backward(ctx: Any, grad_output: Tensor) -> Tuple[Tensor, None]:
        return -ctx.lambda_ * grad_output, None


def apply_gradient_reversal(x: Tensor, lambda_: float) -> Tensor:
    return GradientReversalFunction.apply(x, float(lambda_))


def build_domain2id(sources: List[str]) -> Tuple[Dict[str, int], int]:
    unique = sorted(set(str(s) for s in sources))
    return {s: i for i, s in enumerate(unique)}, len(unique)


class DifficultyClassifierHead(nn.Module):
    def __init__(self, in_dim: int, num_classes: int = 3, dropout: float = 0.1) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class DomainClassifierHead(nn.Module):
    def __init__(self, in_dim: int, num_domains: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_domains),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class ElectraScalarMixDANN(nn.Module):
    def __init__(self, model_name: str, num_classes: int, num_domains: int, dropout: float = 0.2) -> None:
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = int(self.encoder.config.hidden_size)
        n_layers = int(self.encoder.config.num_hidden_layers) + 1
        self.scalar_mix = ScalarMix(n_layers)
        self.dropout = nn.Dropout(dropout)
        self.difficulty_head = DifficultyClassifierHead(hidden, num_classes, dropout)
        self.domain_head = DomainClassifierHead(hidden, num_domains, dropout)

    def encode_pooled(self, input_ids: Tensor, attention_mask: Tensor) -> Tensor:
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        mixed = self.dropout(self.scalar_mix(list(out.hidden_states)))
        mask = attention_mask.unsqueeze(-1).float()
        return (mixed * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

    def forward(self, input_ids: Tensor, attention_mask: Tensor, grl_lambda: float) -> Tuple[Tensor, Tensor]:
        pooled = self.encode_pooled(input_ids, attention_mask)
        diff_logits = self.difficulty_head(pooled)
        dom_logits = self.domain_head(apply_gradient_reversal(pooled, grl_lambda))
        return diff_logits, dom_logits

    def difficulty_logits_only(self, input_ids: Tensor, attention_mask: Tensor) -> Tensor:
        return self.difficulty_head(self.encode_pooled(input_ids, attention_mask))

    def freeze_encoder(self) -> None:
        for p in self.encoder.parameters():
            p.requires_grad = False

    def unfreeze_encoder(self) -> None:
        for p in self.encoder.parameters():
            p.requires_grad = True

    def freeze_encoder_except_top(self, n_layers: int) -> None:
        self.freeze_encoder()
        if hasattr(self.encoder, "encoder") and hasattr(self.encoder.encoder, "layer"):
            for layer in self.encoder.encoder.layer[-n_layers:]:
                for p in layer.parameters():
                    p.requires_grad = True
        for p in self.scalar_mix.parameters():
            p.requires_grad = True


def combined_loss(diff_logits, dom_logits, labels, domains):
    ce_task = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    ce_dom = nn.CrossEntropyLoss()
    diff_loss = ce_task(diff_logits, labels)
    dom_loss = ce_dom(dom_logits, domains)
    return diff_loss + DOMAIN_LOSS_ALPHA * dom_loss, diff_loss, dom_loss

print("DANN model OK")

In [ ]:
# ── Load CSVs from Drive ───────────────────────────────────────────────────
def load_split_csv(name: str) -> pd.DataFrame:
    path = CLEAN_DIR / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    df = df.dropna(subset=[TEXT_COL, LABEL_COL, SOURCE_COL]).reset_index(drop=True)
    df[TEXT_COL] = df[TEXT_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()
    df[SOURCE_COL] = df[SOURCE_COL].astype(str)
    bad = ~df[LABEL_COL].isin(label2id)
    if bad.any():
        print(f"  [{name}] dropping {bad.sum()} rows with unknown labels")
        df = df[~bad].reset_index(drop=True)
    df["label_id"] = df[LABEL_COL].map(label2id).astype(int)
    return df


df_train = load_split_csv("train")
df_val = load_split_csv("val")
df_test = load_split_csv("test")
ood_splits = {
    "ood_onestop": load_split_csv("ood_onestop"),
    "ood_race-middle": load_split_csv("ood_race-middle"),
    "ood_race-high": load_split_csv("ood_race-high"),
}

domain2id, num_domains = build_domain2id(
    pd.concat([df_train[SOURCE_COL], df_val[SOURCE_COL]]).tolist()
)
print(f"num_domains={num_domains}: {list(domain2id.keys())}")


def assign_domain_ids(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    """Map source_dataset → domain_id for DANN training. Drops unknown domains only here."""
    out = df.copy()
    out["domain_id"] = out[SOURCE_COL].map(domain2id)
    unk = out["domain_id"].isna()
    if unk.any():
        print(f"  [{split_name}] dropping {unk.sum()} rows with unknown source_dataset (not in train/val domains)")
        out = out[~unk].reset_index(drop=True)
    out["domain_id"] = out["domain_id"].astype(int)
    return out


# DANN needs domain_id on train/val only. OOD corpora (onestop, race-*) are held out of train
# and must NOT be dropped — eval uses text + judge label only.
df_train = assign_domain_ids(df_train, "train")
df_val = assign_domain_ids(df_val, "val")
# test + OOD: keep all rows; domain_id not used at eval time

if BALANCE_TRAIN:
    min_n = df_train[LABEL_COL].value_counts().min()
    parts = [df_train[df_train[LABEL_COL] == lab].sample(n=min(min_n, len(df_train[df_train[LABEL_COL] == lab])), random_state=RNG_SEED)
             for lab in label2id]
    df_train = pd.concat(parts).sample(frac=1, random_state=RNG_SEED).reset_index(drop=True)

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
print("Train judge labels:\n", df_train[LABEL_COL].value_counts())
for oname, odf in ood_splits.items():
    print(f"{oname}: n={len(odf)}  judge={dict(odf[LABEL_COL].value_counts())}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TextClsDataset(Dataset):
    """Text + difficulty label only (for test / OOD eval — no domain)."""

    def __init__(self, texts: List[str], labels: np.ndarray) -> None:
        self.texts = texts
        self.labels = labels.astype(int)

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        enc = tokenizer(
            self.texts[idx], truncation=True, max_length=MAX_LEN,
            padding="max_length", return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


class TextDANNDataset(Dataset):
    def __init__(self, texts: List[str], labels: np.ndarray, domains: np.ndarray) -> None:
        self.texts = texts
        self.labels = labels.astype(int)
        self.domains = domains.astype(int)

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        enc = tokenizer(
            self.texts[idx], truncation=True, max_length=MAX_LEN,
            padding="max_length", return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
            "domain": torch.tensor(self.domains[idx], dtype=torch.long),
        }


def df_to_dann_loader(df: pd.DataFrame, shuffle: bool) -> DataLoader:
    return DataLoader(
        TextDANNDataset(df[TEXT_COL].tolist(), df["label_id"].values, df["domain_id"].values),
        batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0,
    )


def df_to_cls_loader(df: pd.DataFrame, shuffle: bool) -> DataLoader:
    return DataLoader(
        TextClsDataset(df[TEXT_COL].tolist(), df["label_id"].values),
        batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0,
    )

train_loader = df_to_dann_loader(df_train, shuffle=True)
val_loader = df_to_dann_loader(df_val, shuffle=False)
test_loader = df_to_cls_loader(df_test, shuffle=False)

In [ ]:
# ── Train: Phase 1 (frozen encoder) → Phase 2 (DANN + partial unfreeze) ──
model = ElectraScalarMixDANN(MODEL_NAME, num_classes=3, num_domains=num_domains).to(device)
best_path = os.path.join(DRIVE_OUT_DIR, "best_model.pt")
history: List[Dict] = []
best_val_f1 = -1.0
global_step = 0
total_steps = (PHASE1_EPOCHS + PHASE2_EPOCHS) * len(train_loader)


@torch.no_grad()
def evaluate_val(grl_lambda: float = 0.0) -> Dict[str, float]:
    model.eval()
    ys, preds, ds, dp = [], [], [], []
    for batch in val_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        diff_logits, dom_logits = model(ids, mask, grl_lambda=grl_lambda)
        ys.extend(batch["label"].numpy().tolist())
        preds.extend(diff_logits.argmax(dim=1).cpu().numpy().tolist())
        ds.extend(batch["domain"].numpy().tolist())
        dp.extend(dom_logits.argmax(dim=1).cpu().numpy().tolist())
    return {
        "val_macro_f1": float(f1_score(ys, preds, labels=EVAL_LABELS, average="macro", zero_division=0)),
        "val_acc": float(accuracy_score(ys, preds)),
        "val_domain_acc": float(accuracy_score(ds, dp)),
    }


def run_phase(phase_name: str, epochs: int, lr: float, grl_on: bool) -> None:
    global global_step, best_val_f1
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=min(WARMUP_STEPS, max(1, len(train_loader))),
        num_training_steps=max(1, epochs * len(train_loader)),
    )
    for epoch in range(epochs):
        model.train()
        run_diff, run_dom, run_total, n_batches = 0.0, 0.0, 0.0, 0
        grl_l = 0.0
        for batch in tqdm(train_loader, desc=f"{phase_name} ep{epoch+1}/{epochs}"):
            progress = global_step / max(total_steps - 1, 1)
            grl_l = grl_lambda_schedule(progress) if grl_on else 0.0
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            domains = batch["domain"].to(device)
            diff_logits, dom_logits = model(ids, mask, grl_lambda=grl_l)
            total, diff_loss, dom_loss = combined_loss(diff_logits, dom_logits, labels, domains)
            optimizer.zero_grad()
            total.backward()
            optimizer.step()
            scheduler.step()
            global_step += 1
            run_diff += diff_loss.item()
            run_dom += dom_loss.item()
            run_total += total.item()
            n_batches += 1
        metrics = evaluate_val(grl_lambda=0.0)
        rec = {
            "phase": phase_name, "epoch": epoch + 1, "grl_on": grl_on,
            "grl_lambda": grl_l, "train_diff_loss": run_diff / max(n_batches, 1),
            "train_dom_loss": run_dom / max(n_batches, 1),
            "train_total_loss": run_total / max(n_batches, 1), **metrics,
        }
        history.append(rec)
        print(
            f"{phase_name} ep{epoch+1}: diff={rec['train_diff_loss']:.4f} dom={rec['train_dom_loss']:.4f} "
            f"val_f1={rec['val_macro_f1']:.4f} val_dom_acc={rec['val_domain_acc']:.4f}"
        )
        if metrics["val_macro_f1"] > best_val_f1:
            best_val_f1 = metrics["val_macro_f1"]
            torch.save({"state_dict": model.state_dict(), "domain2id": domain2id}, best_path)
            print(f"  → saved best (val macro-F1 {best_val_f1:.4f})")


model.freeze_encoder()
run_phase("phase1_frozen_encoder", PHASE1_EPOCHS, PHASE1_LR, grl_on=False)

model.unfreeze_encoder()
model.freeze_encoder_except_top(PARTIAL_FREEZE_LAYERS)
run_phase("phase2_dann", PHASE2_EPOCHS, PHASE2_LR, grl_on=True)

ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["state_dict"])
print("Training done. Best val macro-F1:", best_val_f1)

In [ ]:
# ── Eval: test + OOD (judge labels) + confusion matrices ─────────────────

def save_confusion_matrix(y_true, y_pred, title, out_path, labels):
    names = [id2label[i] for i in labels]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True (judge)")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


@torch.no_grad()
def predict_loader(loader: DataLoader) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    ys, preds = [], []
    for batch in loader:
        logits = model.difficulty_logits_only(
            batch["input_ids"].to(device), batch["attention_mask"].to(device)
        )
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        ys.extend(batch["label"].numpy().tolist())
    return np.array(ys), np.array(preds)


@torch.no_grad()
def predict_texts(texts: List[str], batch_size: int = 16) -> np.ndarray:
    model.eval()
    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="predict", leave=False):
        enc = tokenizer(texts[i : i + batch_size], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
        logits = model.difficulty_logits_only(enc["input_ids"].to(device), enc["attention_mask"].to(device))
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    return np.array(preds, dtype=int)


def eval_split(name: str, y_true: np.ndarray, y_pred: np.ndarray) -> Dict:
    if len(y_true) == 0:
        raise ValueError(f"{name}: no samples — check OOD CSVs were not dropped (domain mapping bug)")
    labels = EVAL_LABELS
    present = sorted(set(y_true.tolist()) | set(y_pred.tolist()))
    acc = accuracy_score(y_true, y_pred)
    f1_all = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    f1_present = f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)
    report = classification_report(y_true, y_pred, labels=labels, target_names=[id2label[i] for i in labels], zero_division=0)
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print(report)
    print(f"Accuracy: {acc:.4f}  Macro-F1 (3-class): {f1_all:.4f}  Macro-F1 (present {present}): {f1_present:.4f}")
    cm_path = os.path.join(CM_DIR, f"cm_{name.replace(' ', '_').lower()}.png")
    save_confusion_matrix(y_true, y_pred, name, cm_path, labels)
    print(f"Saved CM → {cm_path}")
    return {
        "corpus": name, "n": int(len(y_true)), "accuracy": float(acc),
        "macro_f1_3class": float(f1_all), "macro_f1_present": float(f1_present),
        "gold_classes": present, "classification_report": report, "confusion_matrix_png": cm_path,
    }


results = {
    "config": {
        "model": MODEL_NAME, "dann": True, "label_col": LABEL_COL,
        "domain_col": SOURCE_COL, "num_domains": num_domains,
        "domain2id": domain2id, "grl_lambda_max": GRL_LAMBDA_MAX,
        "domain_loss_alpha": DOMAIN_LOSS_ALPHA,
        "phase1_epochs": PHASE1_EPOCHS, "phase2_epochs": PHASE2_EPOCHS,
        "best_val_macro_f1": float(best_val_f1), "train_history": history,
    },
    "evaluations": [],
}

y_t, p_t = predict_loader(test_loader)
results["evaluations"].append(eval_split("in_domain_test", y_t, p_t))
for oname, odf in ood_splits.items():
    results["evaluations"].append(eval_split(oname, odf["label_id"].values, predict_texts(odf[TEXT_COL].tolist())))

summary = pd.DataFrame([{"corpus": e["corpus"], "n": e["n"], "accuracy": e["accuracy"],
                         "macro_f1_3class": e["macro_f1_3class"], "macro_f1_present": e["macro_f1_present"]}
                        for e in results["evaluations"]])
print("\nSummary:\n", summary.to_string(index=False))

json_path = os.path.join(DRIVE_OUT_DIR, "eval_results.json")
csv_path = os.path.join(DRIVE_OUT_DIR, "eval_summary.csv")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
summary.to_csv(csv_path, index=False)
print(f"\nSaved JSON → {json_path}")
print(f"Saved CSV  → {csv_path}")
print(f"Model      → {best_path}")